In [ ]:
# Cell 1 - SPEI-3 water scarcity: user inputs and standardized run context
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

from wia_pipelines.config import validate_run_metadata
from wia_pipelines.hazards.spei import SpeiRunInputs, build_spei_run_context, spei_month_window

# USER INPUTS
ISO3 = "HTI"
AS_OF_DATE = "2025-12-31"  # YYYY-MM-DD
LOOKBACK_MONTHS = 12

# SPEI threshold configuration used by downstream mask and aggregation cells
SPEI_THRESHOLDS = {
    "spei3_le_-1_0": -1.0,
    "spei3_le_-1_5": -1.5,
    "spei3_le_-2_0": -2.0,
}
SPEI_VAR = "SPEI3"
DEFAULT_THRESHOLD_KEY = "spei3_le_-1_5"

# Keep this root aligned with historical notebook output locations
OUT_ROOT = Path("./outputs").resolve()

# Build standardized run context and initial schema-compliant metadata
RUN_CTX = build_spei_run_context(
    SpeiRunInputs(
        iso3=ISO3,
        as_of_date=AS_OF_DATE,
        lookback_months=LOOKBACK_MONTHS,
        output_root=OUT_ROOT,
    ),
    create_dirs=True,
    write_metadata=True,
)

# Notebook-level handles reused in later cells
RUN_ID = RUN_CTX["config"].run_id
RUN_METADATA = RUN_CTX["metadata"]
RUN_METADATA_PATH = RUN_CTX["metadata_path"]

# Month window objects used in download and QC steps
WINDOW_INFO = spei_month_window(AS_OF_DATE, lookback_months=LOOKBACK_MONTHS)
WINDOW_MONTHS = [pd.Period(f"{y:04d}-{m:02d}", freq="M") for y, m in WINDOW_INFO["months"]]
WINDOW_START_DATE = WINDOW_MONTHS[0].to_timestamp(how="start").date().isoformat()
WINDOW_END_DATE = WINDOW_MONTHS[-1].to_timestamp(how="end").date().isoformat()

# Store SPEI-specific settings without overwriting schema-required run_config keys
RUN_METADATA["spei_config"] = {
    "pipeline": "water_scarcity_spei3",
    "iso3": ISO3,
    "as_of_date": AS_OF_DATE,
    "window_start_date": WINDOW_START_DATE,
    "window_end_date": WINDOW_END_DATE,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "thresholds": SPEI_THRESHOLDS,
    "default_threshold": DEFAULT_THRESHOLD_KEY,
    "notes": {
        "metric": "SPEI-3 (3-month accumulation), monthly",
        "exposure_rule": "any-month threshold exceedance within the last 12 months",
        "baseline_relative_threshold": False,
        "consecutive_months_required": False,
    },
}
RUN_METADATA["pipeline"] = "water_scarcity_spei3"
validate_run_metadata(RUN_METADATA)
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("ISO3:", ISO3)
print("AS_OF_DATE:", AS_OF_DATE)
print("Monthly window:", f"{WINDOW_START_DATE} -> {WINDOW_END_DATE}", f"({len(WINDOW_MONTHS)} months)")
print("Run ID:", RUN_ID)


In [ ]:
# Cell 2 - Paths, directory aliases, and metadata helpers
from pathlib import Path
import json

from wia_pipelines.core.io_paths import ensure_dir

# COD GDB (admin2 single source of truth)
COD_GDB_ZIP = Path("./data/cod-ab/global_admin_boundaries_matched_latest.gdb.zip").resolve()
COD_LAYER = "admin2"

# WorldPop raster path (country-specific)
worldpop_name = f"{ISO3.lower()}_pop_2025_CN_100m_R2025A_v1.tif"
WORLDPOP_PATH = Path(f"./data/population/{worldpop_name}")

# Basic existence checks
if not COD_GDB_ZIP.exists():
    raise FileNotFoundError(f"Missing admin boundaries GDB zip: {COD_GDB_ZIP}")
if not WORLDPOP_PATH.exists():
    raise FileNotFoundError(f"Missing WorldPop raster: {WORLDPOP_PATH}")

# Backward-compatible DIRS structure expected by downstream notebook cells
RUN_DIR = RUN_CTX["layout"]["base"]
# Canonical run contract directories (shared across hazards)
DIRS = {
    "base": RUN_DIR,
    "raw": RUN_DIR / "raw",
    "intermediate": RUN_DIR / "intermediate",
    "rasters": RUN_DIR / "rasters",
    "tables": RUN_DIR / "tables",
    "qc": RUN_DIR / "qc",
    "logs": RUN_DIR / "logs",

    # Backward-compatible aliases used in existing cells
    "raw_cds": RUN_DIR / "raw" / "cds",
    "raw_extracted": RUN_DIR / "intermediate" / "cds_extracted",
    "masks_native": RUN_DIR / "rasters" / "masks_native",
    "masks_worldpop": RUN_DIR / "rasters" / "masks_worldpop",
    "pop_affected": RUN_DIR / "rasters" / "pop_affected",
}
for p in DIRS.values():
    ensure_dir(p)

CACHE_ROOT = Path("./outputs/_cache").resolve()
CACHE_DIRS = {
    "base": ensure_dir(CACHE_ROOT / "water_scarcity_spei3"),
    "cds_raw": ensure_dir(CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "cds_raw"),
    "cds_extracted": ensure_dir(CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "cds_raw" / "extracted"),
    "aoi": ensure_dir(CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "aoi"),
}

# Metadata helper used by later cells
def update_run_metadata_artifact(key: str, path: Path, note: str | None = None) -> None:
    artifacts = RUN_METADATA.setdefault("artifacts", [])
    record = {"kind": key, "path": str(path), "note": note}
    if isinstance(artifacts, list):
        artifacts.append(record)
    else:
        artifacts[key] = {"path": str(path), "note": note}
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

RUN_METADATA.setdefault("paths", {})
RUN_METADATA["paths"].update(
    {
        "out_root": str(OUT_ROOT),
        "run_dir": str(RUN_DIR),
        "cod_gdb_zip": str(COD_GDB_ZIP),
        "cod_layer": COD_LAYER,
        "cache_root": str(CACHE_ROOT),
    }
)
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Run directory:", RUN_DIR)
print("DIRS keys:", list(DIRS.keys()))
print("CACHE_DIRS keys:", list(CACHE_DIRS.keys()))
print("Run metadata:", RUN_METADATA_PATH)





In [ ]:
# Cell 3 - Load and filter admin2 boundaries, then export QC artifact
from wia_pipelines.hazards.spei import prepare_spei_geography

GEO_CTX = prepare_spei_geography(
    iso3=ISO3,
    admin_path=COD_GDB_ZIP,
    admin_layer=COD_LAYER,
    iso3_field="iso3",
    worldpop_path=WORLDPOP_PATH,
)

admin2_gdf = GEO_CTX["admin_gdf"].copy()

print(f"Loaded admin2 features for {ISO3}: {len(admin2_gdf):,}")
print("CRS:", admin2_gdf.crs)
if "adm2_pcode" in admin2_gdf.columns and len(admin2_gdf) > 0:
    print("Example adm2_pcode:", admin2_gdf["adm2_pcode"].iloc[0])

ADMIN2_GPKG_PATH = DIRS["qc"] / f"{ISO3}_admin2.gpkg"
admin2_gdf.to_file(ADMIN2_GPKG_PATH, layer="admin2", driver="GPKG")
update_run_metadata_artifact(
    "admin2_gpkg", ADMIN2_GPKG_PATH, "Admin2 boundaries filtered to ISO3"
)



In [ ]:
# Cell 4 - Build AOI geometry, CDS request area, and bounds hash
# CDS area order: [North, West, South, East]

import hashlib
from shapely.ops import unary_union

# Country geometry (no artificial buffer) for clipping + admin stats
admin2_4326 = admin2_gdf.to_crs("EPSG:4326").copy()
country_geom_4326 = unary_union(admin2_4326.geometry)

if country_geom_4326.is_empty:
    raise RuntimeError("AOI union geometry is empty -- cannot proceed.")

minx, miny, maxx, maxy = country_geom_4326.bounds
west, south, east, north = float(minx), float(miny), float(maxx), float(maxy)

# CDS fetch buffer: expand by one coarse pixel width (~0.25° for SPEI/UTCI products)
CDS_BUFFER_DEG = 0.25
west_cds = max(-180.0, west - CDS_BUFFER_DEG)
south_cds = max(-90.0, south - CDS_BUFFER_DEG)
east_cds = min(180.0, east + CDS_BUFFER_DEG)
north_cds = min(90.0, north + CDS_BUFFER_DEG)

cds_area = [north_cds, west_cds, south_cds, east_cds]

print("Country bounds (EPSG:4326):")
print(f"  West={west:.4f}, South={south:.4f}, East={east:.4f}, North={north:.4f}")
print("CDS buffer (deg):", CDS_BUFFER_DEG)
print("CDS area:", cds_area)


def bounds_hash_from_area(area: list[float], precision: int = 6) -> str:
    rounded = [round(float(x), precision) for x in area]
    payload = ",".join(f"{x:.{precision}f}" for x in rounded).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def write_bounds_hash(area: list[float], out_path: Path, precision: int = 6) -> str:
    h = bounds_hash_from_area(area, precision=precision)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(h)
    return h


AOI_BOUNDS_HASH_PATH = CACHE_DIRS["aoi"] / f"{ISO3}_admin2_union_bounds_hash.txt"
AOI_HASH = write_bounds_hash(cds_area, AOI_BOUNDS_HASH_PATH)

print("AOI bounds hash:", AOI_HASH)
print("Hash file:", AOI_BOUNDS_HASH_PATH)

RUN_METADATA["aoi"] = {
    "country_bounds_4326": {"west": west, "south": south, "east": east, "north": north},
    "cds_area": cds_area,
    "cds_buffer_deg": CDS_BUFFER_DEG,
    "bounds_hash": AOI_HASH,
    "bounds_hash_path": str(AOI_BOUNDS_HASH_PATH),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))
update_run_metadata_artifact(
    "aoi_bounds_hash",
    AOI_BOUNDS_HASH_PATH,
    "Hash for admin2-union AOI bounds (CDS area)",
)



In [ ]:
# Cell 5 - Validate WorldPop raster and derive valid-population mask
import numpy as np
import rasterio

from wia_pipelines.core.worldpop import worldpop_profile_and_bounds

if not WORLDPOP_PATH.exists():
    raise FileNotFoundError(
        f"WorldPop raster not found: {WORLDPOP_PATH}\n"
        "Set WORLDPOP_PATH to your country-clipped WorldPop GeoTIFF."
    )

wp_info = worldpop_profile_and_bounds(WORLDPOP_PATH)

with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

print("WorldPop path:", WORLDPOP_PATH)
print("WorldPop CRS:", wp_crs)
print("WorldPop shape:", wp_shape)
print("WorldPop nodata:", wp_nodata)
print("WorldPop min/max:", float(np.nanmin(wp_arr)), float(np.nanmax(wp_arr)))
print("WorldPop negatives:", int((wp_arr < 0).sum()))
print("WorldPop bounds (EPSG:4326):", wp_info["bounds_4326"])

pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

total_pop = float(np.nansum(wp_arr[pop_valid_mask]))
print(f"Total population (valid WorldPop pixels): {total_pop:,.0f}")

RUN_METADATA["worldpop"] = {
    "path": str(WORLDPOP_PATH),
    "crs": str(wp_crs),
    "shape": [int(wp_shape[0]), int(wp_shape[1])],
    "nodata": wp_nodata,
    "total_pop_valid": total_pop,
    "bounds_4326": list(wp_info["bounds_4326"]),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))
update_run_metadata_artifact(
    "worldpop_raster",
    WORLDPOP_PATH,
    "Country-clipped WorldPop GeoTIFF (aggregation grid)",
)



In [ ]:
# Cell 6 - Preflight coverage checks before heavy downloads
# Checks:
#   1) WorldPop vs admin bounds
#   2) Single-month SPEI sample coverage vs admin bounds
#   3) Optional quick overlay figures for visual QA

import pandas as pd

from wia_pipelines.hazards.coverage_checks import (
    check_worldpop_coverage,
    plot_grid_overlay_figure,
    plot_raster_overlay_figure,
    run_cds_single_month_check,
    spei_sample_request,
)

# Use first month in the target window for sample download
SAMPLE_YEAR = int(WINDOW_MONTHS[0].year)
SAMPLE_MONTH = int(WINDOW_MONTHS[0].month)

# Admin bounds from Cell 4 (country union, no CDS buffer)
admin_bounds_wsen = (west, south, east, north)

# --- 1) WorldPop coverage check ---
wp_cov = check_worldpop_coverage(admin_bounds_wsen, WORLDPOP_PATH)
print("WorldPop coverage %:", round(wp_cov["coverage_pct"], 3), "full=", wp_cov["full_coverage"])

# --- 2) SPEI single-month sample coverage check ---
preflight_dir = ensure_dir(DIRS["logs"] / "preflight")
sample_zip = preflight_dir / f"{ISO3}_spei_sample_{SAMPLE_YEAR}{SAMPLE_MONTH:02d}.zip"
sample_req = spei_sample_request(SAMPLE_YEAR, SAMPLE_MONTH, cds_area)

sample = run_cds_single_month_check(
    dataset="derived-drought-historical-monthly",
    request=sample_req,
    output_zip=sample_zip,
)
if not sample["ok"]:
    raise RuntimeError(f"SPEI preflight sample download failed: {sample['error']}")

if sample.get("sample_bounds_4326") is None:
    raise RuntimeError("SPEI preflight sample has no inferred bounds.")

from wia_pipelines.core.worldpop import bbox_coverage_report
spei_cov = bbox_coverage_report(admin_bounds_wsen, tuple(sample["sample_bounds_4326"]))
print("SPEI sample coverage %:", round(spei_cov["coverage_pct"], 3), "full=", spei_cov["full_coverage"])

# --- 3) Visual QA figures ---
PRE_QC_DIR = ensure_dir(DIRS["qc"] / "preflight")
spei_nc = Path(sample["first_nc_path"])

wp_raster_fig = PRE_QC_DIR / f"{ISO3}_preflight_worldpop_raster_overlay.png"
wp_grid_fig = PRE_QC_DIR / f"{ISO3}_preflight_worldpop_grid_overlay.png"
spei_raster_fig = PRE_QC_DIR / f"{ISO3}_preflight_spei_raster_overlay.png"
spei_grid_fig = PRE_QC_DIR / f"{ISO3}_preflight_spei_grid_overlay.png"

plot_raster_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    WORLDPOP_PATH,
    "raster",
    wp_raster_fig,
    f"{ISO3} Preflight WorldPop Raster + Admin + AOI",
)
plot_grid_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    WORLDPOP_PATH,
    "raster",
    wp_grid_fig,
    f"{ISO3} Preflight WorldPop Grid + Admin + AOI",
)
plot_raster_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    spei_nc,
    "netcdf",
    spei_raster_fig,
    f"{ISO3} Preflight SPEI Sample Raster + Admin + AOI",
)
plot_grid_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    spei_nc,
    "netcdf",
    spei_grid_fig,
    f"{ISO3} Preflight SPEI Sample Grid + Admin + AOI",
)

RUN_METADATA["preflight_coverage"] = {
    "sample_year": SAMPLE_YEAR,
    "sample_month": SAMPLE_MONTH,
    "worldpop": wp_cov,
    "spei_sample": {
        **sample,
        "coverage_pct": float(spei_cov["coverage_pct"]),
        "full_coverage": bool(spei_cov["full_coverage"]),
    },
    "figures": {
        "worldpop_raster_overlay": str(wp_raster_fig),
        "worldpop_grid_overlay": str(wp_grid_fig),
        "spei_raster_overlay": str(spei_raster_fig),
        "spei_grid_overlay": str(spei_grid_fig),
    },
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

# Fail-fast guard: if sample coverage is not full, don't continue to heavy pulls.
if not spei_cov["full_coverage"]:
    raise RuntimeError(
        f"SPEI preflight coverage is not full ({spei_cov['coverage_pct']:.2f}%). "
        "Adjust CDS buffer and rerun preflight."
    )

print("Preflight coverage checks passed.")



In [ ]:
# Cell 7 - Download SPEI monthly CDS data with consolidated/intermediate fallback + progress
# Dataset: derived-drought-historical-monthly
# Uses CDS area subset from Cell 4: cds_area = [North, West, South, East]

import json
import time
import pandas as pd
from pathlib import Path

from wia_pipelines.core.cds import (
    download_cds,
    ensure_downloads,
    extract_zip_to_dir,
    months_for_last_n,
)

# ---- CDS constants ----
DATASET_SPEI = "derived-drought-historical-monthly"
SPEI_VAR = "standardised_precipitation_evapotranspiration_index"
ACCUMULATION_PERIOD = "3"
VERSION = "1_0"
PRODUCT_TYPE = ["reanalysis"]

DATASET_TYPE_CONS = "consolidated_dataset"
DATASET_TYPE_INT = "intermediate_dataset"

SPEI_DOWNLOAD_STATUS_PATH = DIRS["logs"] / "spei_download_status.json"
_download_t0 = time.perf_counter()


def _write_spei_download_status(*, stage: str, processed: int, total: int, success: int, failed: int, current: str | None = None):
    elapsed = time.perf_counter() - _download_t0
    rate = processed / elapsed if elapsed > 0 else 0.0
    remaining = max(0, total - processed)
    eta = remaining / rate if rate > 0 else None
    payload = {
        "stage": stage,
        "processed": int(processed),
        "total": int(total),
        "success": int(success),
        "failed": int(failed),
        "pct_complete": float((processed / total) * 100.0) if total else 100.0,
        "elapsed_seconds": float(round(elapsed, 2)),
        "rate_items_per_second": float(round(rate, 4)),
        "eta_seconds": None if eta is None else float(round(eta, 2)),
        "current": current,
    }
    SPEI_DOWNLOAD_STATUS_PATH.write_text(json.dumps(payload, indent=2))


def cds_request_spei(area: list[float], year: int, month: int, dataset_type: str) -> dict:
    return {
        "variable": [SPEI_VAR],
        "accumulation_period": [ACCUMULATION_PERIOD],
        "version": VERSION,
        "product_type": PRODUCT_TYPE,
        "dataset_type": dataset_type,
        "year": [str(year)],
        "month": [f"{month:02d}"],
        "area": area,
    }


# ---- Cache key: AOI hash + window end month ----
END_YYYYMM = pd.to_datetime(AS_OF_DATE).strftime("%Y%m")
CACHE_KEY = f"{ISO3}_spei3_{AOI_HASH}_{END_YYYYMM}"

CDS_CACHE_ZIP_DIR = ensure_dir(CACHE_DIRS["cds_raw"])
CDS_CACHE_EXTRACT_DIR = ensure_dir(CACHE_DIRS["cds_extracted"])

# ---- Download month-by-month with fallback ----
months = months_for_last_n(AS_OF_DATE, n_months=12)
manifest = []

n_total = len(months)
n_success = 0
n_failed = 0
_write_spei_download_status(stage="downloads", processed=0, total=n_total, success=0, failed=0)

for idx, (y, m) in enumerate(months, start=1):
    month_label = f"{y}-{m:02d}"
    month_ok = False

    cons_zip = CDS_CACHE_ZIP_DIR / f"{CACHE_KEY}_{y}{m:02d}_{DATASET_TYPE_CONS}.zip"
    if cons_zip.exists() and cons_zip.stat().st_size > 0:
        manifest.append({"year": y, "month": f"{m:02d}", "dataset_type": DATASET_TYPE_CONS, "ok": True, "error": None, "path": str(cons_zip), "cached": True})
        month_ok = True
    else:
        req_cons = cds_request_spei(cds_area, y, m, DATASET_TYPE_CONS)
        ok, err = download_cds(DATASET_SPEI, req_cons, cons_zip)
        manifest.append({"year": y, "month": f"{m:02d}", "dataset_type": DATASET_TYPE_CONS, "ok": ok, "error": err, "path": str(cons_zip), "cached": False})
        if ok:
            month_ok = True
        else:
            int_zip = CDS_CACHE_ZIP_DIR / f"{CACHE_KEY}_{y}{m:02d}_{DATASET_TYPE_INT}.zip"
            if int_zip.exists() and int_zip.stat().st_size > 0:
                manifest.append({"year": y, "month": f"{m:02d}", "dataset_type": DATASET_TYPE_INT, "ok": True, "error": None, "path": str(int_zip), "cached": True})
                month_ok = True
            else:
                req_int = cds_request_spei(cds_area, y, m, DATASET_TYPE_INT)
                ok2, err2 = download_cds(DATASET_SPEI, req_int, int_zip)
                manifest.append({"year": y, "month": f"{m:02d}", "dataset_type": DATASET_TYPE_INT, "ok": ok2, "error": err2, "path": str(int_zip), "cached": False})
                month_ok = bool(ok2)

    if month_ok:
        n_success += 1
    else:
        n_failed += 1

    _write_spei_download_status(
        stage="downloads",
        processed=idx,
        total=n_total,
        success=n_success,
        failed=n_failed,
        current=month_label,
    )

    elapsed = time.perf_counter() - _download_t0
    rate = idx / elapsed if elapsed > 0 else 0.0
    eta = (n_total - idx) / rate if rate > 0 else float("nan")
    print(f"Download progress {idx}/{n_total} ({(idx/n_total)*100:.1f}%) month={month_label} ok={month_ok} eta={eta/60:.1f} min")

# Save/validate manifest
_ = ensure_downloads(
    manifest=manifest,
    kind="spei3",
    logs_dir=DIRS["logs"],
    update_artifact=lambda kind, path, note: update_run_metadata_artifact(kind, path, note),
)

ok_df = pd.DataFrame([m for m in manifest if m.get("ok")])
if ok_df.empty:
    raise RuntimeError("No successful CDS downloads for SPEI-3.")

chosen = (
    ok_df.sort_values(["year", "month", "dataset_type"])
    .groupby(["year", "month"], as_index=False)
    .first()
)

# Extract NetCDFs (cache-aware) with progress
nc_paths = []
extract_total = len(chosen)
extract_ok = 0
extract_fail = 0
for idx, (_, row) in enumerate(chosen.iterrows(), start=1):
    zpath = Path(row["path"])
    extracted = extract_zip_to_dir(zpath, CDS_CACHE_EXTRACT_DIR)
    if not extracted:
        extract_fail += 1
        raise RuntimeError(f"No NetCDF extracted from {zpath}")
    extract_ok += 1
    nc_paths.extend(extracted)

    _write_spei_download_status(
        stage="extract",
        processed=idx,
        total=extract_total,
        success=extract_ok,
        failed=extract_fail,
        current=zpath.name,
    )
    print(f"Extract progress {idx}/{extract_total} ({(idx/extract_total)*100:.1f}%) {zpath.name}")

# Keep unique paths while preserving order
seen = set()
nc_paths_unique = []
for p in nc_paths:
    sp = str(p)
    if sp not in seen:
        seen.add(sp)
        nc_paths_unique.append(Path(sp))
nc_paths = nc_paths_unique

print(f"Selected monthly files: {len(chosen)} (target 12)")
print(f"NetCDF files discovered: {len(nc_paths)}")
if nc_paths:
    print("Example NC:", nc_paths[0])

RUN_METADATA["cds_downloads"] = {
    "dataset": DATASET_SPEI,
    "cache_key": CACHE_KEY,
    "months_target": [f"{y}-{m:02d}" for y, m in months],
    "months_selected": [f"{int(r.year)}-{int(r.month):02d}" for _, r in chosen.iterrows()],
    "n_nc_paths": len(nc_paths),
    "progress_status_path": str(SPEI_DOWNLOAD_STATUS_PATH),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

In [ ]:
# Cell 8 - Open SPEI NetCDF files and build the monthly DataArray stack

import xarray as xr
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
from shapely.geometry import mapping


def find_spei_var(ds: xr.Dataset) -> str:
    if SPEI_VAR in ds.data_vars:
        return SPEI_VAR
    candidates = [v for v in ds.data_vars if v not in ("crs",)]
    if not candidates:
        raise KeyError(f"No data variables found in dataset. Vars={list(ds.data_vars)}")
    return candidates[0]


# ---- Load all NetCDFs and collect DataArrays ----
das = []
for p in nc_paths:
    ds = xr.open_dataset(p, decode_times=True, engine="netcdf4")
    vname = find_spei_var(ds)
    da = ds[vname]

    # Standardise dim names (CDS sometimes uses latitude/longitude)
    rename = {}
    if "latitude" in da.dims:
        rename["latitude"] = "lat"
    if "longitude" in da.dims:
        rename["longitude"] = "lon"
    if rename:
        da = da.rename(rename)

    for d in ("time", "lat", "lon"):
        if d not in da.dims:
            raise KeyError(f"Expected dim '{d}' not found in {p.name}. Dims={da.dims}")

    das.append(da)

# ---- Concatenate and de-duplicate ----
spei = xr.concat(das, dim="time").sortby("time")

tvals = spei["time"].values
_, idx = np.unique(tvals, return_index=True)
spei = spei.isel(time=np.sort(idx)).sortby("time")

# Ensure rioxarray spatial metadata is preserved after concatenation.
# This avoids MissingSpatialDimensionError when clipping and reprojecting downstream.
spei = spei.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
spei = spei.rio.write_crs("EPSG:4326", inplace=False)

# ---- Subset to the required 12 months ----
target_months = [
    str(p) for p in WINDOW_MONTHS
]  # ordered list like ["2024-12", ..., "2025-11"]
spei_months = spei["time"].dt.strftime("%Y-%m").values.tolist()

sel = np.isin(spei_months, target_months)
spei = spei.isel(time=sel).sortby("time")

n_months = int(spei.sizes.get("time", 0))
print(f"SPEI months loaded: {n_months} (expected 12)")

if n_months == 0:
    available = sorted(set(pd.to_datetime(tvals).astype("datetime64[M]").astype(str)))
    raise RuntimeError(
        "No months selected for the target 12-month window.\n"
        f"Target months: {target_months}\n"
        f"Available months in loaded NetCDFs (YYYY-MM): {available[:24]}{' ...' if len(available) > 24 else ''}"
    )

# Report missing months explicitly (helps confirm CDS request completeness)
loaded_months = set(spei["time"].dt.strftime("%Y-%m").values.tolist())
missing_months = [m for m in target_months if m not in loaded_months]
if missing_months:
    print("WARNING: Missing months from target window:", missing_months)

print(
    "First/last timestamp:",
    str(pd.to_datetime(spei["time"].values[0]).date()),
    "→",
    str(pd.to_datetime(spei["time"].values[-1]).date()),
)

# Ensure rioxarray spatial metadata exists on the final object (concat can drop it)
spei = spei.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
spei = spei.rio.write_crs("EPSG:4326", inplace=False)

# Clip to country polygon (admin2 union)
geoms = [mapping(country_geom_4326)]
spei = spei.rio.clip(geoms, crs="EPSG:4326", drop=True, all_touched=True)
spei_da = spei.astype("float32")

RUN_METADATA["spei_data"] = {
    "n_months": n_months,
    "time_start": str(pd.to_datetime(spei_da["time"].values[0]).date()),
    "time_end": str(pd.to_datetime(spei_da["time"].values[-1]).date()),
    "dims": {k: int(v) for k, v in spei_da.sizes.items()},
    "missing_months": missing_months,
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("SPEI DataArray ready:", spei_da)

In [ ]:
# Cell 9 - QC check for NaNs and non-empty values inside the clipped AOI

import numpy as np

total = spei_da.size
nan_count = int(np.isnan(spei_da.values).sum())
finite_count = int(np.isfinite(spei_da.values).sum())

print(f"Total cells (time*lat*lon): {total:,}")
print(f"NaNs: {nan_count:,} ({nan_count/total:.1%})")
print(f"Finite: {finite_count:,} ({finite_count/total:.1%})")

# Check at least some finite data exists each month
finite_by_month = (
    np.isfinite(spei_da.values).reshape(spei_da.sizes["time"], -1).sum(axis=1)
)
print("Finite cells per month:", finite_by_month.tolist())

# Summary stats over finite values only (across all months)
vals = spei_da.values[np.isfinite(spei_da.values)]
print(
    "Finite SPEI3 min/median/max:",
    float(np.min(vals)),
    float(np.median(vals)),
    float(np.max(vals)),
)


In [ ]:
# Cell 10 - Build SPEI masks, reproject to WorldPop grid, and write affected-pop rasters + progress
# Output artifacts (GeoTIFFs on WorldPop grid):
#   - masks_worldpop/spei_mask_<key>.tif            (uint8: 1=exceedance, 0=no exceedance, 255=nodata)
#   - pop_exposed/spei_pop_affected_<key>.tif       (float32, WorldPop counts where mask==1 else 0)

import json
import time
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin

from wia_pipelines.core.raster_ops import reproject_array_to_grid, write_array_geotiff

SPEI_THRESHOLDS = {
    "rel_spei_le_m1p0": -1.0,
    "rel_spei_le_m1p5": -1.5,
    "rel_spei_le_m2p0": -2.0,
}
DEFAULT_SPEI_KEY = "rel_spei_le_m1p5"

MASKS_NATIVE_DIR = ensure_dir(DIRS["masks_native"] / "spei")
MASKS_WP_DIR = ensure_dir(DIRS["masks_worldpop"] / "spei")
POP_AFFECTED_DIR = ensure_dir(DIRS["pop_affected"] / "spei")
QC_DIR = ensure_dir(DIRS["qc"] / "spei_masks")

SPEI_RASTER_STATUS_PATH = DIRS["logs"] / "spei_raster_compute_status.json"
_raster_t0 = time.perf_counter()


def _write_spei_raster_status(*, processed: int, total: int, current: str | None = None):
    elapsed = time.perf_counter() - _raster_t0
    rate = processed / elapsed if elapsed > 0 else 0.0
    remaining = max(0, total - processed)
    eta = remaining / rate if rate > 0 else None
    payload = {
        "stage": "raster_compute",
        "processed": int(processed),
        "total": int(total),
        "pct_complete": float((processed / total) * 100.0) if total else 100.0,
        "elapsed_seconds": float(round(elapsed, 2)),
        "rate_items_per_second": float(round(rate, 4)),
        "eta_seconds": None if eta is None else float(round(eta, 2)),
        "current": current,
    }
    SPEI_RASTER_STATUS_PATH.write_text(json.dumps(payload, indent=2))


def spei_any_month_mask(spei_da: xr.DataArray, thr: float) -> xr.DataArray:
    cond = spei_da <= thr
    cond = cond.fillna(False)
    return cond.any(dim="time")


def _mask_da_transform(mask_da_2d: xr.DataArray):
    lats = mask_da_2d["lat"].values
    lons = mask_da_2d["lon"].values
    dlat = float(np.abs(lats[1] - lats[0]))
    dlon = float(np.abs(lons[1] - lons[0]))
    north = float(np.max(lats) + dlat / 2.0)
    west = float(np.min(lons) - dlon / 2.0)
    return from_origin(west, north, dlon, dlat)


with rasterio.open(WORLDPOP_PATH) as wp:
    wp_profile = wp.profile
    wp_profile.update(count=1)
    wp_arr = wp.read(1).astype("float64")
    wp_nodata = wp.nodata

pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

mask_products = {}
threshold_items = list(SPEI_THRESHOLDS.items())
_write_spei_raster_status(processed=0, total=len(threshold_items), current=None)

for idx, (key, thr) in enumerate(threshold_items, start=1):
    mask_native_da = spei_any_month_mask(spei_da, thr)
    native_path = MASKS_NATIVE_DIR / f"spei_mask_{key}_native.tif"

    src_mask_arr = mask_native_da.values.astype(np.uint8)
    src_transform = _mask_da_transform(mask_native_da)

    write_array_geotiff(native_path, src_mask_arr, transform=src_transform, crs="EPSG:4326", nodata=255, dtype="uint8")

    mask_wp = reproject_array_to_grid(
        src_array=src_mask_arr,
        src_transform=src_transform,
        src_crs="EPSG:4326",
        dst_shape=(wp_profile["height"], wp_profile["width"]),
        dst_transform=wp_profile["transform"],
        dst_crs=wp_profile["crs"],
        src_nodata=255,
        dst_nodata=255,
        resampling="nearest",
    ).astype(np.uint8)

    mask_wp_path = MASKS_WP_DIR / f"spei_mask_{key}_worldpop.tif"
    write_array_geotiff(mask_wp_path, mask_wp, transform=wp_profile["transform"], crs=wp_profile["crs"], nodata=255, dtype="uint8")

    pop_affected = np.zeros_like(wp_arr, dtype=np.float32)
    affected = (mask_wp == 1) & pop_valid_mask
    pop_affected[affected] = wp_arr[affected].astype(np.float32)

    pop_path = POP_AFFECTED_DIR / f"spei_pop_affected_{key}.tif"
    write_array_geotiff(pop_path, pop_affected, transform=wp_profile["transform"], crs=wp_profile["crs"], nodata=0.0, dtype="float32")

    pop_sum = float(np.nansum(pop_affected))
    mask_products[key] = {
        "threshold": float(thr),
        "mask_native_path": str(native_path),
        "mask_worldpop_path": str(mask_wp_path),
        "pop_affected_path": str(pop_path),
        "pop_affected_sum": pop_sum,
    }

    _write_spei_raster_status(processed=idx, total=len(threshold_items), current=key)
    elapsed = time.perf_counter() - _raster_t0
    rate = idx / elapsed if elapsed > 0 else 0.0
    eta = (len(threshold_items) - idx) / rate if rate > 0 else float("nan")
    print(f"Raster progress {idx}/{len(threshold_items)} ({(idx/len(threshold_items))*100:.1f}%) key={key} pop_affected={pop_sum:,.0f} eta={eta:.1f}s")

RUN_METADATA["spei_masks"] = {
    "thresholds": {k: float(v) for k, v in SPEI_THRESHOLDS.items()},
    "default_key": DEFAULT_SPEI_KEY,
    "products": mask_products,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "aggregation_rule": "any_month_spei_le_threshold",
    "progress_status_path": str(SPEI_RASTER_STATUS_PATH),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

update_run_metadata_artifact(
    "spei_default_mask_worldpop",
    Path(mask_products[DEFAULT_SPEI_KEY]["mask_worldpop_path"]),
    "Default SPEI-3 drought mask on WorldPop grid (uint8: 1=exceedance, 0=no exceedance, 255=nodata)",
)
update_run_metadata_artifact(
    "spei_default_pop_affected",
    Path(mask_products[DEFAULT_SPEI_KEY]["pop_affected_path"]),
    "Default SPEI-3 population affected raster on WorldPop grid (float32 counts)",
)

In [ ]:
# Cell 11 - Aggregate affected population to admin2 and compute percentages
# Produces: <ISO3>_admin2_water_scarcity_spei3_<AS_OF_DATE>.csv
#
# Uses WorldPop grid as common raster:
#   - WORLDPOP_PATH (country-clipped WorldPop)
#   - POP_AFFECTED_DIR / spei_pop_affected_<key>.tif (from Cell 8)
#
# Output columns:
#   adm2_pcode, pop_total,
#   pop_affected_<key>, pct_affected_<key>  for each threshold key

import pandas as pd
import rasterio
from rasterio.features import rasterize
import numpy as np

# ---- Load admin2 boundaries (filtered earlier to ISO3) ----
admin2 = admin2_gdf.copy()

if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Available columns: {list(admin2.columns)}")

admin2 = (
    admin2[["adm2_pcode", "geometry"]]
    .dropna(subset=["adm2_pcode", "geometry"])
    .reset_index(drop=True)
)

# ---- Open WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

# Robust validity mask (exclude nodata + negatives)
pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

# Reproject admin2 to match WorldPop CRS
admin2_wp = admin2.to_crs(wp_crs)

# ---- Rasterise admin2 polygons to unique IDs ----
shapes = [(geom, i + 1) for i, geom in enumerate(admin2_wp.geometry)]
admin2_id_raster = rasterize(
    shapes=shapes,
    out_shape=wp_shape,
    transform=wp_transform,
    fill=0,
    dtype="int32",
    all_touched=True,
)

flat_ids = admin2_id_raster.ravel()
flat_pop = wp_arr.ravel()
flat_pop_valid = pop_valid_mask.ravel()

valid = (flat_ids > 0) & flat_pop_valid
ids_valid = flat_ids[valid]
pop_valid = flat_pop[valid]

# Total pop by admin2 id (denominator)
pop_total_by_id = np.bincount(
    ids_valid, weights=pop_valid, minlength=len(admin2_wp) + 1
)

# Scaffold output table
out = pd.DataFrame(
    {
        "adm2_pcode": admin2_wp["adm2_pcode"].values,
        "admin2_id": np.arange(1, len(admin2_wp) + 1, dtype=int),
    }
)
out["pop_total"] = out["admin2_id"].map(
    lambda i: float(pop_total_by_id[i]) if i < len(pop_total_by_id) else 0.0
)

# ---- Aggregate affected population per threshold ----
for key in SPEI_THRESHOLDS.keys():
    pop_path = POP_AFFECTED_DIR / f"spei_pop_affected_{key}.tif"
    if not pop_path.exists():
        raise FileNotFoundError(f"Missing pop affected raster for {key}: {pop_path}")

    with rasterio.open(pop_path) as src:
        arr = src.read(1).astype("float64")
        arr_nodata = src.nodata

        # Grid safety check
        if arr.shape != wp_shape:
            raise ValueError(
                f"Shape mismatch for {key}: {arr.shape} vs WorldPop {wp_shape}"
            )

        arr_ok = np.isfinite(arr)
        if arr_nodata is not None:
            arr_ok &= arr != float(arr_nodata)
        arr_ok &= arr >= 0

        flat_arr = arr.ravel()
        valid_arr = valid & arr_ok.ravel()

        ids_valid_arr = flat_ids[valid_arr]
        vals_valid_arr = flat_arr[valid_arr]

        pop_aff_by_id = np.bincount(
            ids_valid_arr, weights=vals_valid_arr, minlength=len(admin2_wp) + 1
        )

    col_pop = f"pop_affected_{key}"
    col_pct = f"pct_affected_{key}"

    out[col_pop] = out["admin2_id"].map(
        lambda i: float(pop_aff_by_id[i]) if i < len(pop_aff_by_id) else 0.0
    )
    out[col_pct] = np.where(
        out["pop_total"] > 0, (out[col_pop] / out["pop_total"]) * 100.0, np.nan
    )

# ---- Save table ----
OUT_CSV_PATH = DIRS["tables"] / f"{ISO3}_admin2_water_scarcity_spei3_{AS_OF_DATE}.csv"
out = out.drop(columns=["admin2_id"])
out.to_csv(OUT_CSV_PATH, index=False)

update_run_metadata_artifact(
    "admin2_water_scarcity_table",
    OUT_CSV_PATH,
    "Admin2 population % affected by water scarcity (SPEI-3 thresholds; any month in 12-month window)",
)

print("Wrote:", OUT_CSV_PATH)
print(out.head(10).to_string(index=False))

RUN_METADATA["admin2_table_spei"] = {
    "path": str(OUT_CSV_PATH),
    "n_admin2": int(len(out)),
    "thresholds": {k: float(v) for k, v in SPEI_THRESHOLDS.items()},
    "default_key": DEFAULT_SPEI_KEY,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "rule": "any_month_spei_le_threshold",
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

In [ ]:
# Cell 12 - QC and figures using windowed sums and downsampled plotting
# Ensure WorldPop NoData and negative values are masked before plotting.

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
from pathlib import Path

QC_SPEI_DIR = ensure_dir(DIRS["qc"] / "spei")
FIG_DIR = ensure_dir(QC_SPEI_DIR / "figures")

mask_wp_path = MASKS_WP_DIR / f"spei_mask_{DEFAULT_SPEI_KEY}_worldpop.tif"
pop_aff_path = POP_AFFECTED_DIR / f"spei_pop_affected_{DEFAULT_SPEI_KEY}.tif"

if not mask_wp_path.exists():
    raise FileNotFoundError(mask_wp_path)
if not pop_aff_path.exists():
    raise FileNotFoundError(pop_aff_path)


def windowed_sum_worldpop(path: Path) -> tuple[float, dict]:
    """Sum valid WorldPop values using block windows (excludes nodata and negatives)."""
    total = 0.0
    with rasterio.open(path) as src:
        nodata = src.nodata
        profile = src.profile
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window).astype("float64", copy=False)
            m = np.isfinite(arr)
            if nodata is not None:
                m &= arr != float(nodata)
            m &= arr >= 0
            if m.any():
                total += float(arr[m].sum())
    return total, profile


def windowed_sum_masked_pop(worldpop_path: Path, mask_path: Path) -> float:
    """Sum WorldPop where mask==1, using block windows (constant memory)."""
    total = 0.0
    with rasterio.open(worldpop_path) as wp, rasterio.open(mask_path) as ms:
        if (wp.height != ms.height) or (wp.width != ms.width):
            raise ValueError("WorldPop and mask shapes differ.")
        if wp.crs != ms.crs:
            raise ValueError("WorldPop and mask CRS differ.")
        if not wp.transform.almost_equals(ms.transform):
            raise ValueError("WorldPop and mask transforms differ.")

        wp_nodata = wp.nodata
        mask_nodata = ms.nodata

        for _, window in wp.block_windows(1):
            pop = wp.read(1, window=window).astype("float64", copy=False)
            mask = ms.read(1, window=window).astype("uint8", copy=False)

            pop_ok = np.isfinite(pop)
            if wp_nodata is not None:
                pop_ok &= pop != float(wp_nodata)
            pop_ok &= pop >= 0

            mask_ok = mask == 1
            if mask_nodata is not None:
                mask_ok &= mask != int(mask_nodata)

            sel = pop_ok & mask_ok
            if sel.any():
                total += float(pop[sel].sum())
    return total


def windowed_sum_pop_affected(pop_aff_path: Path) -> float:
    """Sum affected raster values using block windows (excludes nodata and negatives)."""
    total = 0.0
    with rasterio.open(pop_aff_path) as src:
        nodata = src.nodata
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window).astype("float64", copy=False)
            m = np.isfinite(arr)
            if nodata is not None:
                m &= arr != float(nodata)
            m &= arr >= 0
            if m.any():
                total += float(arr[m].sum())
    return total


def read_overview(path: Path, max_dim: int = 2000, resampling=Resampling.nearest):
    """
    Read a downsampled overview for plotting.
    Returns (arr, extent, meta).
    """
    with rasterio.open(path) as src:
        h, w = src.height, src.width
        scale = max(h / max_dim, w / max_dim, 1.0)
        out_h = int(np.ceil(h / scale))
        out_w = int(np.ceil(w / scale))

        arr = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
        )

        transform = src.transform * src.transform.scale((w / out_w), (h / out_h))
        extent = plotting_extent(arr, transform)
        meta = {
            "crs": src.crs,
            "transform": transform,
            "nodata": src.nodata,
            "shape": (out_h, out_w),
        }
    return arr, extent, meta


# ---- Sanitisation for plotting (avoid nodata dominating colour scale) ----
def sanitise_worldpop_for_plot(arr: np.ndarray, nodata) -> np.ndarray:
    a = arr.astype("float64", copy=False)
    bad = ~np.isfinite(a)
    if nodata is not None:
        bad |= a == float(nodata)
    bad |= a < 0
    a = a.copy()
    a[bad] = np.nan
    return a


def sanitise_pop_affected_for_plot(arr: np.ndarray, nodata) -> np.ndarray:
    a = arr.astype("float64", copy=False)
    bad = ~np.isfinite(a)
    if nodata is not None:
        bad |= a == float(nodata)
    bad |= a < 0
    a = a.copy()
    a[bad] = np.nan
    return a


def sanitise_mask_for_plot(arr: np.ndarray) -> np.ndarray:
    # keep only 0/1; everything else -> nan so colourbar isn't weird
    a = arr.astype("float64", copy=False)
    bad = ~np.isin(a, [0, 1])
    a = a.copy()
    a[bad] = np.nan
    return a


# ---- Admin2 boundaries ----
admin2 = admin2_gdf.copy()
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_crs = wp.crs
admin2_wp = admin2.to_crs(wp_crs)

# Simplify for speed
admin2_wp_simpl = admin2_wp.copy()
admin2_wp_simpl["geometry"] = admin2_wp_simpl.geometry.simplify(
    tolerance=0.002, preserve_topology=True
)

# ---- QC metrics (constant memory) ----
worldpop_total_pop, wp_profile = windowed_sum_worldpop(WORLDPOP_PATH)
mask_pop_sum = windowed_sum_masked_pop(WORLDPOP_PATH, mask_wp_path)
pop_affected_sum = windowed_sum_pop_affected(pop_aff_path)

with rasterio.open(WORLDPOP_PATH) as wp, rasterio.open(
    mask_wp_path
) as ms, rasterio.open(pop_aff_path) as pa:
    grid_ok = (
        (wp.height == ms.height == pa.height)
        and (wp.width == ms.width == pa.width)
        and (wp.crs == ms.crs == pa.crs)
        and wp.transform.almost_equals(ms.transform)
        and wp.transform.almost_equals(pa.transform)
    )
    wp_nodata = wp.nodata
    pop_aff_nodata = pa.nodata
    wp_shape = (wp.height, wp.width)

qc = {
    "iso3": ISO3,
    "as_of_date": AS_OF_DATE,
    "window_months": ", ".join([str(p) for p in WINDOW_MONTHS]),
    "default_threshold_key": DEFAULT_SPEI_KEY,
    "default_threshold_value": float(SPEI_THRESHOLDS[DEFAULT_SPEI_KEY]),
    "worldpop_total_pop": worldpop_total_pop,
    "mask_pop_sum": mask_pop_sum,
    "pop_affected_sum": pop_affected_sum,
    "mask_vs_pop_affected_diff": (mask_pop_sum - pop_affected_sum),
    "grid_alignment_ok": bool(grid_ok),
    "worldpop_shape": str(wp_shape),
    "worldpop_crs": str(wp_crs),
    "worldpop_nodata": wp_nodata,
    "pop_aff_nodata": pop_aff_nodata,
}
qc_df = pd.DataFrame([qc])
QC_CSV = QC_SPEI_DIR / f"{ISO3}_spei_qc_{AS_OF_DATE}.csv"
qc_df.to_csv(QC_CSV, index=False)
update_run_metadata_artifact(
    "spei_qc_table", QC_CSV, "QC summary for SPEI water scarcity processing"
)
print(qc_df.T.to_string(header=False))

# ---- FIG 1: WorldPop overview ----
wp_over, wp_extent, wp_meta = read_overview(
    WORLDPOP_PATH, max_dim=2000, resampling=Resampling.average
)
wp_over = sanitise_worldpop_for_plot(wp_over, wp_meta["nodata"])

fig1, ax1 = plt.subplots(figsize=(8, 6))
im1 = ax1.imshow(wp_over, extent=wp_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax1, linewidth=0.3, edgecolor="black")
ax1.set_title(f"{ISO3} WorldPop overview with Admin2 outlines")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
FIG1 = FIG_DIR / f"{ISO3}_fig1_worldpop_{AS_OF_DATE}.png"
fig1.savefig(FIG1, dpi=200, bbox_inches="tight")
plt.close(fig1)

# ---- FIG 2: SPEI mask overview ----
mask_over, mask_extent, _ = read_overview(
    mask_wp_path, max_dim=2000, resampling=Resampling.nearest
)
mask_over = sanitise_mask_for_plot(mask_over)

fig2, ax2 = plt.subplots(figsize=(8, 6))
im2 = ax2.imshow(mask_over, extent=mask_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax2, linewidth=0.3, edgecolor="black")
ax2.set_title(f"{ISO3} SPEI-3 mask ({DEFAULT_SPEI_KEY}) overview (WorldPop grid)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
FIG2 = FIG_DIR / f"{ISO3}_fig2_spei_mask_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig2.savefig(FIG2, dpi=200, bbox_inches="tight")
plt.close(fig2)

# ---- FIG 3: Pop affected overview ----
pa_over, pa_extent, pa_meta = read_overview(
    pop_aff_path, max_dim=2000, resampling=Resampling.average
)
pa_over = sanitise_pop_affected_for_plot(pa_over, pa_meta["nodata"])

fig3, ax3 = plt.subplots(figsize=(8, 6))
im3 = ax3.imshow(pa_over, extent=pa_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax3, linewidth=0.3, edgecolor="black")
ax3.set_title(f"{ISO3} Population affected overview (SPEI-3 {DEFAULT_SPEI_KEY})")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
FIG3 = FIG_DIR / f"{ISO3}_fig3_pop_affected_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig3.savefig(FIG3, dpi=200, bbox_inches="tight")
plt.close(fig3)

# ---- FIG 4: Admin2 choropleth (% affected) ----
pct_col = f"pct_affected_{DEFAULT_SPEI_KEY}"
admin2_join = admin2_wp.merge(
    out[["adm2_pcode", "pop_total", pct_col]], on="adm2_pcode", how="left"
)

fig4, ax4 = plt.subplots(figsize=(8, 6))
admin2_join.plot(column=pct_col, ax=ax4, legend=True)
admin2_wp_simpl.boundary.plot(ax=ax4, linewidth=0.3, edgecolor="black")
ax4.set_title(f"{ISO3} Admin2 % population affected (SPEI-3 {DEFAULT_SPEI_KEY})")
ax4.set_axis_off()
FIG4 = FIG_DIR / f"{ISO3}_fig4_admin2_pct_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig4.savefig(FIG4, dpi=200, bbox_inches="tight")
plt.close(fig4)

RUN_METADATA["spei_qc_figures"] = {
    "qc_csv": str(QC_CSV),
    "fig_worldpop": str(FIG1),
    "fig_mask": str(FIG2),
    "fig_pop_affected": str(FIG3),
    "fig_admin2_choropleth": str(FIG4),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Saved figures:")
print(" -", FIG1)
print(" -", FIG2)
print(" -", FIG3)
print(" -", FIG4)

In [ ]:
# Cell 13 - Export QC summary and figures to a single PDF report
# Uses matplotlib PdfPages for a simple, dependable multi-page report.

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

PDF_PATH = QC_SPEI_DIR / f"{ISO3}_spei_qc_report_{AS_OF_DATE}.pdf"

# Reload QC table for insertion into the PDF summary page
qc_df = pd.read_csv(QC_CSV)

with PdfPages(PDF_PATH) as pdf:
    # Page 1: QC text summary
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait
    ax.axis("off")

    lines = []
    lines.append(f"SPEI-3 Water Scarcity QC Report — {ISO3}")
    lines.append(f"As-of date: {AS_OF_DATE}")
    lines.append(f"Window months: {qc_df.loc[0,'window_months']}")
    lines.append("")
    lines.append(
        f"Default threshold: {qc_df.loc[0,'default_threshold_key']}  (<= {qc_df.loc[0,'default_threshold_value']})"
    )
    lines.append("")
    lines.append("Key checks:")
    lines.append(f"- Grid alignment OK: {bool(qc_df.loc[0,'grid_alignment_ok'])}")
    lines.append(
        f"- WorldPop total population: {qc_df.loc[0,'worldpop_total_pop']:,.0f}"
    )
    lines.append(
        f"- Masked population (WorldPop sum where mask==1): {qc_df.loc[0,'mask_pop_sum']:,.0f}"
    )
    lines.append(f"- Pop affected raster sum: {qc_df.loc[0,'pop_affected_sum']:,.0f}")
    lines.append(
        f"- Difference (mask pop - pop affected): {qc_df.loc[0,'mask_vs_pop_affected_diff']:,.0f}"
    )
    lines.append("")
    lines.append("Raster metadata:")
    lines.append(f"- WorldPop shape: {qc_df.loc[0,'worldpop_shape']}")
    lines.append(f"- WorldPop CRS: {qc_df.loc[0,'worldpop_crs']}")
    lines.append(f"- WorldPop NoData: {qc_df.loc[0,'worldpop_nodata']}")
    lines.append(f"- Pop affected NoData: {qc_df.loc[0,'pop_aff_nodata']}")

    ax.text(0.05, 0.95, "\n".join(lines), va="top", ha="left", fontsize=11)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # Pages 2-5: QC figures
    for fig_path in [FIG1, FIG2, FIG3, FIG4]:
        img = plt.imread(fig_path)
        fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape-ish
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(Path(fig_path).name, fontsize=10)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

update_run_metadata_artifact(
    "spei_qc_pdf", PDF_PATH, "QC report PDF for SPEI water scarcity (figures + summary)"
)

RUN_METADATA["spei_qc_report_pdf"] = {"path": str(PDF_PATH)}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Wrote PDF:", PDF_PATH)

In [ ]:
# Cell 14 - Run standardized post-run validation (schema + parity report)
import json
import subprocess
import sys
from pathlib import Path

run_dir = Path(RUN_CTX["layout"]["base"]).resolve()
cmd = [
    sys.executable,
    "scripts/spei_postrun.py",
    "--run-dir",
    str(run_dir),
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    raise RuntimeError("SPEI post-run checks failed.")

summary = json.loads(result.stdout)
print("Post-run status:", summary["status"])
print("Parity report:", summary["outputs"].get("parity_md"))
